In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\ADYA TRIPATHI\OneDrive\Desktop\ML Projects\MicroGuard\smart_manufacturing_data.csv")

In [3]:
df.shape

(100000, 13)

In [4]:
df.head()

,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,failure_type,downtime_risk,maintenance_required
0,2025-01-01 00:00:00,39,78.61,28.65,79.96,3.73,2.16,1,0,106,Normal,0.0,0
1,2025-01-01 00:01:00,29,68.19,57.28,35.94,3.64,0.69,1,0,320,Normal,0.0,0
2,2025-01-01 00:02:00,15,98.94,50.20,72.06,1.00,2.49,1,1,19,Normal,1.0,1
3,2025-01-01 00:03:00,43,90.91,37.65,30.34,3.15,4.96,1,1,10,Normal,1.0,1
4,2025-01-01 00:04:00,8,72.32,40.69,56.71,2.68,0.63,2,0,65,Vibration Issue,0.0,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   timestamp                 100000 non-null  object 
 1   machine_id                100000 non-null  int64  
 2   temperature               100000 non-null  float64
 3   vibration                 100000 non-null  float64
 4   humidity                  100000 non-null  float64
 5   pressure                  100000 non-null  float64
 6   energy_consumption        100000 non-null  float64
 7   machine_status            100000 non-null  int64  
 8   anomaly_flag              100000 non-null  int64  
 9   predicted_remaining_life  100000 non-null  int64  
 10  failure_type              100000 non-null  object 
 11  downtime_risk             100000 non-null  float64
 12  maintenance_required      100000 non-null  int64  
dtypes: float64(6), int64(5), object(2)
memory usa

In [6]:
print(df['machine_status'].value_counts())
print(df['anomaly_flag'].value_counts())

machine_status
1    80091
2    10057
0     9852
Name: count, dtype: int64
anomaly_flag
0    91084
1     8916
Name: count, dtype: int64


In [7]:
print(df['failure_type'].value_counts())

failure_type
Normal              91899
Vibration Issue      3129
Overheating          1989
Pressure Drop        1969
Electrical Fault     1014
Name: count, dtype: int64


In [8]:
print(df.groupby('machine_status')[['anomaly_flag','downtime_risk','maintenance_required']].mean())		

                anomaly_flag  downtime_risk  maintenance_required
machine_status                                                   
0                   0.090438       0.090438              0.109216
1                   0.088662       0.088657              0.106928
2                   0.091876       0.091864              1.000000


In [13]:
print(df.groupby('machine_status')['predicted_remaining_life'].describe())	

                  count        mean         std  min   25%    50%    75%  \
machine_status                                                             
0                9852.0  234.548011  150.271367  1.0  96.0  232.0  366.0   
1               80091.0  234.380442  149.979427  1.0  97.0  230.0  365.0   
2               10057.0  233.109774  150.533903  1.0  94.0  230.0  365.0   

                  max  
machine_status         
0               499.0  
1               499.0  
2               499.0  


In [12]:
print(pd.crosstab(df['machine_status'], df['failure_type']))

failure_type    Electrical Fault  Normal  Overheating  Pressure Drop  \
machine_status                                                         
0                              0    9852            0              0   
1                              0   80091            0              0   
2                           1014    1956         1989           1969   

failure_type    Vibration Issue  
machine_status                   
0                             0  
1                             0  
2                          3129  


According to the data for machine status 
0 must be the running machines because has low anomaly rate and low downtime risk and lowest maintainence required ,these machines will be used to define the normal behaviour for our modelso that it can differentiate b/w normal and anomaly behaviour
1 must be the warning running machines because has low anomaly rate and low downtime risk and lowest maintainence required 
2 must be the machines which fail often because the crosstab shows a lot of values in every failure type

In [19]:
cols = ['timestamp','machine_id','temperature','vibration','humidity','pressure','energy_consumption','machine_status','anomaly_flag']
df_core = df[cols] 

For anomaly detection the normal baehaviour will be used for training so that the maodel will take the normal data and learn its properties like mean and covariance and establish a normal behaviour boundary and when queried it can clearly flag the query 

In [20]:
normal_data = df['machine_status'] == 0

In [21]:
# training sirf normal par 
X_train = df.loc[normal_data,cols].values

In [23]:
# testing sab par
X_test = df[cols].values
# model ki performance test krne ke liye
y_test = df['anomaly_flag'].values